# Gaussian process

In this notebook we will use the GP to model synthetic datasets.

In [ ]:
%%capture
!pip install git+https://github.com/italo-goncalves/geoML.git@claude

In [ ]:
import geoml
import geoml.kernels as kr
import geoml.transform as tr

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 1-dimensional case

First we generate the dataset. We will intentionally leave a gap in the middle and add some noise.

In [ ]:
np.random.seed(0)
n_samples = 100

X = np.concatenate([
    np.random.uniform(-5, -1, int(n_samples/2))[:, None],
    np.random.uniform(1, 5, int(n_samples/2))[:, None]
])
y = np.cos(1.5 * X) + np.sin(5 * X) + np.random.normal(0, 0.2, n_samples)[:, None]

plt.figure(figsize=(10, 5))
plt.scatter(X, y, c='k')
plt.xlabel("X")
plt.ylabel("y")
plt.title("Synthetic Dataset")
plt.show()

In `geoML` the dependent variable `y` is treated as a random variable. There are specialized data objects to deal with this situation.

In [ ]:
points = geoml.data.PointData.from_array(X)
points

Now we add the measurements of `y`.

In [ ]:
points.add_continuous_variable(name='y', measurements=y)
points

### The GP model

Now that the data is set, we can build the model. The transform parameter `r` is the range. It is initialized with an arbitrary value to be optimized during training.

In [ ]:
cov = kr.Covariance(
    kernel=kr.Gaussian(),
    transform=tr.Isotropic(r=1.0)
)

model_1d = geoml.models.GP(
    data=points,
    variable='y',
    covariance=cov
)

model_1d.train(max_iter=300)

plt.plot(model_1d.training_log)
plt.xlabel("Iteration")
plt.ylabel("Log-likelihood")
plt.show()

We can see the values of the parameters after training.

In [ ]:
model_1d

The learned noise level:

In [ ]:
model_1d.parameters['noise'].get_value().numpy()

To make predictions, first we define a grid of coordinates.

In [ ]:
grid_1d = geoml.data.Grid1D(start=-8, end=8, n=1001)

model_1d.predict(grid_1d)

Now we can plot the results. The `geoML` data objects store the predicted mean, variance, and percentiles.

The `reset_quantiles` method copmute the values corresponding to the percentiles that we want to see in the model's output. We are going to work with the median and a 50% and 95% confidence intervals.

In [ ]:
grid_1d.variables['y'].reset_quantiles((0.025, 0.25, 0.5, 0.75, 0.975))

plt.figure(figsize=(15, 5))
plt.fill_between(
    grid_1d.coordinates[:, 0],
    grid_1d.variables['y'].quantiles[0.975].values,
    grid_1d.variables['y'].quantiles[0.025].values,
    alpha=0.3, color='dodgerblue',
    label='95% confidence interval'
)
plt.fill_between(
    grid_1d.coordinates[:, 0],
    grid_1d.variables['y'].quantiles[0.75].values,
    grid_1d.variables['y'].quantiles[0.25].values,
    alpha=0.3, color='dodgerblue',
    label='50% confidence interval'
)
plt.plot(
    grid_1d.coordinates,
    grid_1d.variables['y'].quantiles[0.5].values,
    color='dodgerblue',
    label='Median'
)
plt.scatter(X, y, c='k', label='Training data')
plt.xlabel("X")
plt.ylabel("y")
plt.legend(loc='upper right')
plt.show()

It can be seen that the prediction quickly reverts to 0 and the uncertainty rises to the maximum value as we move away from the data. Note that there is always a minimum amount of uncertainty due to the noise in the data.

## 2-dimensional case

We generate a synthetic dataset, just as above.

In [ ]:
# Generate grid of points
n_points = 201
x = np.linspace(-3, 3, n_points)
y = np.linspace(-3, 3, n_points)
X, Y = np.meshgrid(x, y)

# Create a function with many maxima and minima
Z = (np.sin(3 * X) * np.cos(3 * Y) +
     0.5 * np.sin(5 * X + Y) -
     0.3 * np.cos(X - 4 * Y))

# Normalize to the range [-2, 2]
Z_min, Z_max = Z.min(), Z.max()
Z_normalized = 4 * (Z - Z_min) / (Z_max - Z_min) - 2

# Visualize
plt.figure(figsize=(6, 5))
plt.imshow(Z_normalized, extent=(-3, 3, -3, 3), origin='lower', cmap='RdYlBu')
plt.colorbar(label='Z')
plt.title('Synthetic 2D Dataset')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

Let us extract a subset of this data to use as training data.

In [ ]:
df = pd.DataFrame(
    {
        'x': X.flatten(),
        'y': Y.flatten(),
        'z': Z_normalized.flatten()
    }
)

df = df.sample(200)
df

For the sake of realism we add a small amount of noise to the data.

In [ ]:
noisy_z = df['z'] + np.random.normal(0, 0.02, df.shape[0])

points_2d = geoml.data.PointData(df, coordinates=['x', 'y'])
points_2d.add_continuous_variable(name='z', measurements=noisy_z)
points_2d

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(df['x'], df['y'], c=df['z'], cmap='RdYlBu')
plt.colorbar(label='Z')

### The GP model

In [ ]:
cov = kr.Covariance(
    kernel=kr.Gaussian(),
    transform=tr.Isotropic(r=1.0)
)

model_2d = geoml.models.GP(
    data=points_2d,
    variable='z',
    covariance=cov
)

model_2d.train(max_iter=1000)

plt.plot(model_2d.training_log)
plt.xlabel("Iteration")
plt.ylabel("Log-likelihood")
plt.show()

In [ ]:
model_2d

As before, we generate a grid to receive the predictions.

In [ ]:
grid_2d = geoml.data.Grid2D(
    start=(-3, -3),
    end=(3, 3),
    n=(n_points, n_points)
)

model_2d.predict(grid_2d)

In `geoML` each random variable has a number of `Attributes` that are used to represent a statistic such as the mean, variance, a given percentile, etc. The attributes are aware of their corresponding spatial object. In the case of a 2D grid, the `as_image()` method converts it to a matrix for easy plotting.

In [ ]:
grid_2d.variables['z'].reset_quantiles((0.025, 0.25, 0.5, 0.75, 0.975))

pred_median = grid_2d.variables['z'].quantiles[0.5].as_image()

pred_conf = grid_2d.variables['z'].quantiles[0.975].as_image() \
          - grid_2d.variables['z'].quantiles[0.025].as_image()

error = pred_median - Z_normalized
max_error = np.max(np.abs(error))

fig, ax = plt.subplots(1, 3, figsize=(18, 4))

ax[0].imshow(pred_median, extent=(-3, 3, -3, 3), origin='lower', cmap='RdYlBu',
             vmin=-2, vmax=2)
ax[0].set_title('Predicted median')
ax[1].imshow(pred_conf, extent=(-3, 3, -3, 3), origin='lower', cmap='turbo')
ax[1].set_title('95% confidence interval')
ax[2].imshow(error, extent=(-3, 3, -3, 3), vmin=-max_error, vmax=max_error,
             origin='lower', cmap='RdYlBu')
ax[2].scatter(df['x'], df['y'], c='k', s=0.1)
ax[2].set_title('Error')
plt.colorbar(ax[0].images[0], ax=ax[0])
plt.colorbar(ax[1].images[0], ax=ax[1])
plt.colorbar(ax[2].images[0], ax=ax[2])
plt.show()

It can be seen that the true function was well modelled with a relatively small number of data points.